# Day 5: MLP Credit Scoring

**Goal:** Train a multi-layer perceptron (MLP) to predict whether a loan application is approved.

**Dataset:** 500 synthetic applicants, 5 features, binary label (1 = approved, 0 = denied)  
**Connection to Days 3-4:** Logistic Regression -> Decision Tree -> MLP (same task, richer model)


In [ ]:
# Cell 0: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

NAVY  = '#1e3a5f'
AMBER = '#d97706'
SLATE = '#64748b'


## The Five Features: A Credit Analyst Reads Them

Before writing any code, understand what each column means in finance terms:

| Feature | Finance meaning | What it measures |
|---|---|---|
| `income` | **Capacity to repay** | Higher income = more buffer to service debt |
| `debt_ratio` | **Existing obligations** | Fraction of income already committed to other debt |
| `credit_score` | **Track record** | History of paying on time (300=poor, 850=excellent) |
| `employment_years` | **Stability** | Job tenure reduces income interruption risk |
| `loan_amount` | **Exposure** | Larger loan = larger potential loss for the lender |

The approval rule built into this dataset: **credit score > 650 AND debt ratio < 0.45 AND income > 45k**.
The MLP does not know this rule. It has to discover it from 500 labelled examples.


In [ ]:
# Cell 1: Generate synthetic credit dataset (n=500)
n = 500

income           = np.random.uniform(20_000, 120_000, n)
debt_ratio       = np.random.uniform(0.1, 0.9, n)
credit_score     = np.random.uniform(300, 850, n)
employment_years = np.random.uniform(0, 20, n)
loan_amount      = np.random.uniform(5_000, 50_000, n)

# Approval rule: score > 650 AND debt < 0.45 AND income > 45k, plus 8% noise
base  = ((credit_score > 650) & (debt_ratio < 0.45) & (income > 45_000)).astype(int)
noise = np.random.binomial(1, 0.08, n)
y     = np.where(noise, 1 - base, base)

feature_names = ['income', 'debt_ratio', 'credit_score', 'employment_years', 'loan_amount']
X = np.column_stack([income, debt_ratio, credit_score, employment_years, loan_amount])

df = pd.DataFrame(X, columns=feature_names)
df['approved'] = y


In [ ]:
# Cell 2: Describe data and check class balance
print('Feature statistics:')
print(df[feature_names].describe().round(1))
print()

n_approved = y.sum()
n_denied   = n - n_approved
print(f'Class balance: {n_approved} approved ({n_approved/n:.1%})  |  {n_denied} denied ({n_denied/n:.1%})')


## CHECKPOINT: Predict the Leakage Consequence

**Before running the scaler, discuss with your neighbour:**

Suppose we fit `StandardScaler` on the **entire** dataset (all 500 rows) before splitting.
The scaler then computes the mean and standard deviation including the 100 test rows.

**Predict:** Why is this a problem? What unfair advantage does the model gain?

Your prediction: ________

*(The next two cells show the wrong way and why it matters.)*


In [ ]:
# [MIN]
# WRONG WAY: fit StandardScaler on ALL data before splitting -- data leakage!
from sklearn.preprocessing import StandardScaler as _SC
from sklearn.model_selection import train_test_split as _tts

scaler_wrong = _SC()
X_all_scaled = scaler_wrong.fit_transform(X)   # sees test rows during fit!

Xtr_w, Xte_w, ytr_w, yte_w = _tts(X_all_scaled, y, test_size=0.2, random_state=42)

# The scaler now 'knows' statistics from the test set.
# Even before training, test data has leaked into the preprocessing step.
print(f'Scaler mean (income, WRONG -- includes test rows): {scaler_wrong.mean_[0]:.1f}')
print('This mean was computed using the 100 test applicants.')
print('In production there are no test rows -- you would never have this information.')


## Why Data Leakage Inflates Your Accuracy

When the scaler sees test rows during fit, it encodes test-set statistics into the
feature representation. The model indirectly 'sees' the test data before evaluation,
so reported accuracy is optimistically biased.

**Fix (Cells 3 and 4 below):** split FIRST, then fit the scaler ONLY on training rows.
Use `scaler.transform(X_test)` (not `fit_transform`) on the test set so it uses only
the training distribution for standardization.

> **Finance analogy:** A credit model trained on today's customers must not peek at
> tomorrow's applicants when setting its scoring thresholds. The scaler is part of the
> model -- it must be 'frozen' at training time.


In [ ]:
# Cell 3: Train / test split 80 / 20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Train: {X_train.shape[0]} samples   Test: {X_test.shape[0]} samples')


In [ ]:
# Cell 4: StandardScaler -- fit ONLY on training data
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform train
X_test_sc  = scaler.transform(X_test)        # transform test with train's parameters

print('Scaling complete.')
print(f'Train mean (income col): {X_train_sc[:, 0].mean():.4f}  (should be ~0)')
print(f'Train std  (income col): {X_train_sc[:, 0].std():.4f}   (should be ~1)')


Each training step begins with a **forward pass** (compute predictions) followed by **backpropagation** (update weights). Together they form the learning loop.

## How an MLP Makes a Prediction

An MLP works in two phases every training step:

**Forward pass:** Data flows left to right through the network.
Each neuron computes a weighted sum of its inputs, applies a non-linear activation (ReLU),
and passes the result to the next layer. The final layer outputs class probabilities.

**Backpropagation:** The network compares its prediction to the true label.
It computes the gradient of the loss with respect to every weight, then nudges each
weight in the direction that reduces the loss (gradient descent).

Repeat for 500 epochs. That is the full training loop.

> **Why not logistic regression?** Logistic regression draws a single straight line.
> The MLP stacks non-linear transformations, so it can learn curved decision boundaries
> -- useful when approval depends on interactions between income AND debt ratio.


In [ ]:
# [FULL]
# INSTRUCTOR DEMO: forward pass by hand -- 1 input, 1 hidden neuron, ReLU

# Toy network: 1 input feature -> 1 hidden neuron (ReLU) -> 1 output (sigmoid)
x_in  = 0.7       # normalised income of one applicant
w1    = 0.5       # weight: input -> hidden
b1    = -0.2      # bias of hidden neuron
w2    = 1.3       # weight: hidden -> output
b2    = 0.1       # bias of output neuron

# Hidden layer
z1    = w1 * x_in + b1
a1    = max(0, z1)    # ReLU activation

# Output layer
z2    = w2 * a1 + b2
prob  = 1 / (1 + np.exp(-z2))    # sigmoid -> probability of approval

print(f'Input (normalised income):     {x_in:.2f}')
print(f'Hidden pre-activation z1:     {z1:.2f}')
print(f'Hidden post-ReLU a1:          {a1:.2f}')
print(f'Output pre-activation z2:     {z2:.2f}')
print(f'Predicted approval prob:       {prob:.2%}')
print()
print('Backpropagation would then compare this prob to the true label')
print('and nudge w1, b1, w2, b2 to reduce the error.')


## CHECKPOINT: Predict the MLP Accuracy

**Before training the MLP, discuss with your neighbour:**

Our dataset has a clear rule: approve if credit score > 650 AND debt ratio < 0.45 AND income > 45k.
There is 8% label noise (some correct decisions are randomly flipped).

**Predict:** What test accuracy do you expect the MLP to achieve?

- Below 75%? (worse than a naive majority-class predictor)
- 75-85%? (decent, close to the noise ceiling)
- Above 90%? (very high for noisy data -- is this plausible?)

Your prediction: ________

*(Run Cell 5 to train and Cell 6 to evaluate.)*


## Connection to Day 4: How the MLP Learns

Remember from Day 4: the MLP adjusts its weights using **backpropagation**.  
Each epoch, the network makes predictions, computes the loss (how wrong it was),  
and propagates the error backwards through every layer to update the weights.  

Our architecture: **input (5) -> hidden (64) -> hidden (32) -> output (2)**  
Each hidden layer applies a non-linear activation (ReLU by default), which lets the  
network learn non-linear decision boundaries that logistic regression cannot capture.


In [ ]:
# Cell 5: Train MLP classifier
clf = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    random_state=42
)
clf.fit(X_train_sc, y_train)
print(f'Training stopped at iteration: {clf.n_iter_}')
print(f'Final training loss: {clf.loss_curve_[-1]:.4f}')


In [ ]:
# Cell 6: Evaluate -- accuracy, confusion matrix, classification report
y_pred = clf.predict(X_test_sc)

print(f'Test accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print('Confusion Matrix (rows=actual, cols=predicted):')
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['Denied', 'Approved']))


In [ ]:
# Cell 7: Loss curve -- shows training convergence
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(clf.loss_curve_, color=NAVY, linewidth=2)
ax.set_xlabel('Iteration (epoch)')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('MLP Training Loss Curve', color=NAVY)
ax.set_yscale('log')
plt.tight_layout()
plt.show()
print(f'Loss went from {clf.loss_curve_[0]:.4f} to {clf.loss_curve_[-1]:.4f}')


## Connection to Day 4: What the Loss Curve Shows

This curve is what **gradient descent minimizes** during training.  
Each point on the x-axis is one pass through the entire training set (an epoch).  
The steep early drop means the network is learning fast; the flattening means it has  
converged to a local minimum. If the curve is still dropping when training stops,  
you may benefit from increasing `max_iter`.


In [ ]:
# Cell 8: Guided experiment -- compare architectures
# Change hidden_layer_sizes and observe the effect on accuracy and convergence speed.

architectures = [
    (16,),          # very small: one hidden layer, 16 neurons
    (64, 32),       # baseline (what we trained above)
    (128, 64, 32),  # deeper: three hidden layers
]

print(f'{"Architecture":25s}  {"Test acc":>10s}  {"Iterations":>12s}')
print('-' * 52)
for arch in architectures:
    m = MLPClassifier(hidden_layer_sizes=arch, max_iter=500, random_state=42)
    m.fit(X_train_sc, y_train)
    acc = m.score(X_test_sc, y_test)
    print(f'{str(arch):25s}  {acc:>10.3f}  {m.n_iter_:>12d}')

# Questions to discuss:
# 1. Does the bigger network always win? Why or why not?
# 2. Which architecture converges fastest (fewest iterations)?
# 3. How would your answer change if n=50 instead of n=500?
